> ## ⚠️  Superseded — single-run draft, retained as a record
>
> **This notebook will not run.** It opens `processed_data/all_data.pkl`, the earlier
> 53-feature preprocessing artefact that was replaced by `canonical_data.pkl` when the
> feature set was corrected. Nothing in this folder regenerates it, so the first cell raises
> `FileNotFoundError`.
>
> **No result reported in the thesis comes from this notebook.** Every reported figure is
> produced by the canonical path `00`/`01` → `08`–`15`, which trains over thirty fixed seeds
> (`SEEDS = [42 + 7*i for i in range(30)]`). Model definitions — `TemporalAttention`,
> `build_lstm_transformer`, `build_standalone_lstm` — live in `canonical_pipeline.py`.
>
> It is kept because its saved outputs are the record of the single-run baseline stage the
> project passed through. See README §5.

# 02 — Train SARIMA
Statistical baseline model (Section 2.2.1)

In [1]:
import pickle, pandas as pd, numpy as np, warnings
warnings.filterwarnings('ignore')
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

with open('processed_data/all_data.pkl', 'rb') as f:
    data = pickle.load(f)
train_df = data['train_df']
y_test = data['y_test']
test_df = data['test_df']
print(f"Train: {len(train_df)} days, Test: {len(test_df)} days")

Train: 2888 days, Test: 723 days


In [2]:
print("Training SARIMA (this may take 1-2 minutes)...")
sarima_model = auto_arima(
    train_df['dengue_cases'], seasonal=True, m=7,
    max_p=2, max_q=2, max_P=1, max_Q=1, max_d=1, max_D=1, D=0,
    trace=False, error_action='ignore', suppress_warnings=True, stepwise=True)

sarima_pred = sarima_model.predict(n_periods=len(test_df))
rmse = np.sqrt(mean_squared_error(y_test, sarima_pred))
mae = mean_absolute_error(y_test, sarima_pred)
r2 = r2_score(y_test, sarima_pred)

print(f"\n✅ SARIMA — RMSE: {rmse:.2f}, MAE: {mae:.2f}, R²: {r2:.4f}")
print(f"   Order: {sarima_model.order}, Seasonal: {sarima_model.seasonal_order}")

Training SARIMA (this may take 1-2 minutes)...

✅ SARIMA — RMSE: 62.52, MAE: 37.71, R²: -0.2567
   Order: (2, 1, 2), Seasonal: (1, 0, 1, 7)


In [3]:
# Save
results = {'model': 'SARIMA', 'RMSE': rmse, 'MAE': mae, 'R2': r2, 'predictions': sarima_pred}
with open('metrics/sarima_results.pkl', 'wb') as f:
    pickle.dump(results, f)
with open('models/sarima_model.pkl', 'wb') as f:
    pickle.dump(sarima_model, f)
print("✅ Saved to models/ and metrics/")

✅ Saved to models/ and metrics/
